## Load Raw Data

Read the World Bank PPI database (2010–2024, energy sector) from the raw Excel file. The `CustomQuery` sheet contains one row per project.

In [5]:
import pandas as pd

df = pd.read_excel(
    '../../data/raw/WB_PPI_2010-2024_energy.xlsx',
    sheet_name='CustomQuery'
)

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 3 rows:')
df.head(3)

Shape: (3008, 45)

Columns: ['Region', 'Country', 'IncomeGroup', 'IDA Status', 'Financial closure year', 'Financial closure Month', 'Project name', 'RelatedNames', 'Type of PPI', 'Subtype of PPI', 'Project status', 'Primary sector', 'Subsector', 'Segment', 'Location', 'ContractPeriod', 'GovtGrantingContract', 'DirectGovtSupport', 'DirectGovtSupportValue', 'InDirectGovtSupport', 'InDirectGovtSupportValue', 'Total Equity', 'InvestmentYear', 'PercentPrivate', 'FeesToGovernment', 'PhysicalAssets', 'TotalInvestment', 'CapacityType', 'Capacity', 'Technology', 'RelatedProjects', 'BidCriteria', 'AwardMethod', 'NumberOfBids', 'Sponsors', 'Sponsors Country', 'Main Revenue Source', 'Other Revenue Source', 'MultiLateralSupport', 'BiLateralSupport', 'TotalDebtFunding', 'DebtEquityGrantRatio', 'ProjectBanks', 'UnsolicitedProposal', 'PublicDisclosure']

First 3 rows:


,Region,Country,IncomeGroup,IDA Status,Financial closure year,Financial closure Month,Project name,RelatedNames,Type of PPI,Subtype of PPI,...,Sponsors Country,Main Revenue Source,Other Revenue Source,MultiLateralSupport,BiLateralSupport,TotalDebtFunding,DebtEquityGrantRatio,ProjectBanks,UnsolicitedProposal,PublicDisclosure
0,East Asia and Pacific,Cambodia,Low income,IDA,2010,February,Cambodia Energy Limited,Leader - CIID,Greenfield project,"Build, own, and operate",...,..,Purchase agreements or transmission fees with ...,NaN,No,No,NaN,NaN,EXIM Bank Malaysia (Not Available / Not Availa...,NaN,No
1,East Asia and Pacific,Cambodia,Low income,IDA,2010,February,North Phnom Penh - Kampong Power Transmission,Cambodia Transmission Limited,Greenfield project,"Build, operate, and transfer",...,..,Purchase agreements or transmission fees with ...,NaN,MIGA (Guarantee / $73 Million / 2019),No,Not Applicable,70/30,EXIM Bank Malaysia (Not Available / Internatio...,No,No
2,East Asia and Pacific,Cambodia,Low income,IDA,2010,July,Orussei Hydroelectric Power Plant,Orussey,Greenfield project,"Build, operate, and transfer",...,China,Purchase agreements or transmission fees with ...,NaN,No,No,NaN,NaN,EX-IM Bank of China (Not Available / Not Avail...,NaN,No


## Filter to Electricity Subsector

The raw data covers all energy PPI projects (electricity, pipelines, natural gas distribution, etc.). We keep only rows where `Subsector == 'Electricity'` since the analysis focuses on electricity generation and transmission investment.

In [6]:
print('Shape before filter:', df.shape)

df_elec = df[df['Subsector'] == 'Electricity'].copy()

print('Shape after filter: ', df_elec.shape)
print('\nTechnology value_counts:')
print(df_elec['Technology'].value_counts())

Shape before filter: (3008, 45)
Shape after filter:  (2951, 45)

Technology value_counts:
Technology
Solar, PV                         919
Wind                              738
Hydro, Small (<50MW)              238
Not Applicable                    212
Biomass                           188
Natural Gas                       133
Hydro, Large (>50MW)              127
Coal                              107
Waste                              65
Diesel                             45
Geothermal                         30
Other                              25
Biogas                             24
Solar, CSP                         21
Solar, PV, N/A                     20
Solar, CPV                         10
Wind, N/A                           4
Wind, Solar, PV                     4
Solar, PV, Wind                     4
Solar, PV, Solar, PV                3
Wind, Not Applicable                2
Natural Gas, Diesel                 2
Natural Gas, Steam                  2
Diesel, Natural Gas      

## Drop Uninformative Technology Values

Rows where `Technology` is `'Not Applicable'` or `'Other'` carry no classifiable signal — we cannot determine whether the project is green or brown. These are dropped before classification. Null values are also removed for the same reason.

In [7]:
print('Shape before drop:', df_elec.shape)

exclude = {'Not Applicable', 'Other'}
df_elec = df_elec[
    ~df_elec['Technology'].isin(exclude) &
    df_elec['Technology'].notna()
].copy()

print('Shape after drop: ', df_elec.shape)
print('\nTechnology value_counts:')
pd.set_option('display.max_rows', None)
print(df_elec['Technology'].value_counts())

Shape before drop: (2951, 45)
Shape after drop:  (2699, 45)

Technology value_counts:
Technology
Solar, PV                         919
Wind                              738
Hydro, Small (<50MW)              238
Biomass                           188
Natural Gas                       133
Hydro, Large (>50MW)              127
Coal                              107
Waste                              65
Diesel                             45
Geothermal                         30
Biogas                             24
Solar, CSP                         21
Solar, PV, N/A                     20
Solar, CPV                         10
Wind, N/A                           4
Wind, Solar, PV                     4
Solar, PV, Wind                     4
Solar, PV, Solar, PV                3
Wind, Not Applicable                2
Natural Gas, Diesel                 2
Natural Gas, Steam                  2
Diesel, Natural Gas                 2
Solar, PV, Wind, N/A                2
Wind, Other                  

## Green / Brown Classification

We classify each project into one of three categories based on the `Technology` string.

**Why classify at all?** The research question is about the direction of private investment — are countries moving capital toward low-carbon electricity or locking in fossil fuels? That requires a binary green/brown signal at the project level.

**Green (is_green = 1):** Technologies with no meaningful direct carbon emissions from generation — solar (all variants), wind, small hydro (<50MW), geothermal, biomass, and biogas. Multi-technology strings composed entirely of green technologies (e.g. `Solar, PV, Wind`) are also classified green. Strings where the only non-green element is `N/A` or `Not Applicable` are treated as green since they indicate no real secondary technology.

**Brown (is_green = 0):** Fossil fuel technologies — coal, natural gas, diesel, steam, and their brown-only combinations.

**Why are Hydro Large (>50MW) and Waste left as ambiguous (NaN)?**
- *Hydro Large* provides carbon-free generation but is associated with major ecological disruption (reservoir flooding, altered river flow) and contested social impacts. Its inclusion would substantially shift the green share for countries like Albania or Cambodia that have large hydro assets.
- *Waste-to-energy* reduces landfill but combusts organic material, giving it a mixed carbon profile that does not fit cleanly into either category.

Rather than making an arbitrary call, both are left as NaN in the primary `is_green` column and excluded from the main analysis.

**Why create two versions (narrow and broad)?** Excluding 192 projects is a real analytical choice that affects results. The `is_green_broad` column reclassifies Hydro Large and Waste as green (1), so we can run the full analysis under both definitions and compare. If conclusions hold under both, the classification call does not drive the finding.

**The 6 noisy hybrid rows** (e.g. `Wind, Coal, N/A`, `Natural Gas, Steam, Solar, CSP`) mix green and brown technologies with no clean assignment. At 6 rows total they are negligible and are left as NaN in both definitions.

In [8]:
import numpy as np

GREEN = {
    # Pure solar
    'Solar, PV', 'Solar, CSP', 'Solar, CPV',
    # Solar + no meaningful second tech
    'Solar, PV, N/A', 'Solar, PV, Not Applicable',
    # Pure wind
    'Wind',
    # Wind + no meaningful second tech
    'Wind, N/A', 'Wind, Not Applicable',
    # Green-only combos
    'Wind, Solar, PV', 'Solar, PV, Wind', 'Solar, PV, Wind, N/A',
    'Solar, PV, Solar, PV', 'Solar, PV, Biogas',
    # Other explicitly green
    'Hydro, Small (<50MW)', 'Geothermal', 'Biomass', 'Biogas',
}

BROWN = {
    'Coal', 'Natural Gas', 'Diesel', 'Steam',
    # Brown-only combos
    'Natural Gas, Diesel', 'Diesel, Natural Gas',
    'Natural Gas, Steam',
}

# Treated as green only under the broad definition
GREEN_BROAD_EXTRA = {'Hydro, Large (>50MW)', 'Waste'}

def classify(tech):
    if tech in GREEN:
        return 1, 0
    elif tech in BROWN:
        return 0, 0
    else:
        return np.nan, 1  # ambiguous or noisy hybrid

df_elec[['is_green', 'ambiguous_flag']] = (
    df_elec['Technology'].apply(lambda t: pd.Series(classify(t)))
)

# Broad: same as narrow but Hydro Large and Waste count as green
df_elec['is_green_broad'] = df_elec['is_green'].copy()
df_elec.loc[df_elec['Technology'].isin(GREEN_BROAD_EXTRA), 'is_green_broad'] = 1

print('is_green value_counts (including NaN):')
print(df_elec['is_green'].value_counts(dropna=False))

print('\nis_green_broad value_counts (including NaN):')
print(df_elec['is_green_broad'].value_counts(dropna=False))

print('\nAmbiguous Technology values:')
for v in sorted(df_elec.loc[df_elec['ambiguous_flag'] == 1, 'Technology'].unique()):
    n = (df_elec['Technology'] == v).sum()
    print(f'  {n:>4}  {v}')

is_green value_counts (including NaN):
is_green
1.0    2209
0.0     292
NaN     198
Name: count, dtype: int64

is_green_broad value_counts (including NaN):
is_green_broad
1.0    2401
0.0     292
NaN       6
Name: count, dtype: int64

Ambiguous Technology values:
   127  Hydro, Large (>50MW)
     1  Natural Gas, Other
     1  Natural Gas, Steam, Solar, CSP
     1  Solar, PV, Other
     1  Solar, PV, Wind, Coal, N/A
    65  Waste
     1  Wind, Coal, N/A
     1  Wind, Other


In [9]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

df = pd.read_csv('../../data/clean/ppi_green_share.csv')
df.head(20)

,Country,Financial closure year,total_investment,total_projects,green_investment,green_projects,green_share_value,green_share_count,total_investment_broad,total_projects_broad,green_investment_broad,green_projects_broad,green_share_value_broad,green_share_count_broad
0,Afghanistan,2017,19.000,1.000,19.000,1.000,1.000,1.000,19.000,1,19.000,1.000,1.000,1.000
1,Afghanistan,2019,190.490,3.000,18.890,1.000,0.099,0.333,190.490,3,18.890,1.000,0.099,0.333
2,Albania,2011,4.160,1.000,4.160,1.000,1.000,1.000,4.160,1,4.160,1.000,1.000,1.000
3,Albania,2012,134.400,7.000,134.400,7.000,1.000,1.000,1504.400,8,1504.400,8.000,1.000,1.000
4,Albania,2013,19.400,1.000,19.400,1.000,1.000,1.000,19.400,1,19.400,1.000,1.000,1.000
5,Albania,2014,177.300,1.000,177.300,1.000,1.000,1.000,177.300,1,177.300,1.000,1.000,1.000
6,Albania,2023,187.330,2.000,187.330,2.000,1.000,1.000,187.330,2,187.330,2.000,1.000,1.000
7,Algeria,2012,30.300,1.000,30.300,1.000,1.000,1.000,30.300,1,30.300,1.000,1.000,1.000
8,Angola,2014,112.000,1.000,112.000,1.000,1.000,1.000,112.000,1,112.000,1.000,1.000,1.000
9,Angola,2024,1600.000,1.000,1600.000,1.000,1.000,1.000,1600.000,1,1600.000,1.000,1.000,1.000
